In [35]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = 'data/messy_mashup'
WORKING_BASE = 'data/milestone4'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')

def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)

def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")

Q1) You ran the generate_synthetic_dataset script with samples_per_genre=50. If the script executed successfully across all 10 target genres, exactly how many .wav files should now exist in your /kaggle/working/synthetic_mashups/train/ directory tree?

In [5]:
# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)

In [12]:
p = Path("data/milestone4/synthetic_mashups/train")
len([f for f in p.rglob("*") if f.is_file()])

500

Q2) Load any of your newly generated .wav files using torchaudio.load(). Given our configuration (target_sr=22050, duration=30), what is the exact tensor shape of the resulting waveform?
Provide in tuple format, example (x,y)

In [15]:
torchaudio.load("data/milestone4/synthetic_mashups/train/blues/mashup_000.wav")[0].shape

torch.Size([2, 661500])

Q3) Using the extract_and_save_features script with n_fft=2048, hop_length=512, and n_mels=128, you converted the waveforms into .pt files. 

If you load one of these pre-computed tensors using torch.load(), what is its exact dimension shape?

In [36]:
INPUT_DIR = 'data/milestone4/synthetic_mashups/train'
OUTPUT_DIR = 'data/milestone4/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to data/milestone4/features/train


In [45]:
torch.load("data/milestone4/features/train/blues/mashup_000.pt").shape

torch.Size([2, 128, 1292])

Build Your Model (The CRNN)
Goal: You will build a model that understands both what sounds are playing (using a CNN) and the rhythm of how they play over time (using an RNN).

What to Code:

Create a PyTorch class called CRNN with these exact pieces:

1. The Input Your model will take in a 1-channel Mel-spectrogram (treat it like a single-color image).

2. The CNN (Sound Feature Extractor)

Write two blocks of layers to find patterns in the audio frequencies:

Block 1: Conv2D (32 filters, $3 \times 3$ kernel, padding 1) $\rightarrow$ BatchNorm2d $\rightarrow$ ReLU $\rightarrow$ MaxPool2d ($2 \times 2$ window).

Block 2: Conv2D (64 filters, $3 \times 3$ kernel, padding 1) ---> BatchNorm2d ---> ReLU ---> MaxPool2d ($2 \times 2$ window).

3. The Bridge (Reshaping)

The CNN outputs a 3D block of data (channels, frequencies, and time). To pass this into an RNN, you must reshape it. Keep the time steps as your sequence, and flatten the channels and frequencies together into one single feature list per time step.

4. The RNN (Rhythm Tracker)

Pass your reshaped sequence into a 1-layer Bidirectional LSTM. (Set hidden_size=64 and batch_first=True).

5. The Final Prediction

The LSTM outputs data for every single time step. To get one final answer for the whole 30-second song, take the maximum value across all time steps (known as Global Max Pooling). Finally, pass that summary vector through a standard Linear layer to output the 10 genre scores.

In [96]:
from torch import nn
from torch.utils.data import Dataset

class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        # Extract genre from the directory name
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre] # converts genre name to a numerically encoded value
        
        # Load precomputed tensor
        feature = torch.load(file_path)
        return feature, label

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        """
        Input: (B, 1, 128, T)
        """
        super().__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 2))
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 2))
        )
        
        self.lstm = nn.LSTM(
            input_size=64*32,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        """
        Input:
            x: Tensor of shape (Batch, 1, 128, Time) representing Mel-spectrograms.
        Output:
            logits: Tensor of shape (Batch, num_classes) representing unnormalized class scores.
        """
        print(x.shape)
        x = self.block1(x)
        print(x.shape)
        x = self.block2(x)
        print(x.shape)

        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, t, c*f)
        print(x.shape)
        
        lstm_out, _ = self.lstm(x)
        print(lstm_out.shape)
        max_vals, max_idxs = torch.max(lstm_out, dim=1)
        print(max_vals.shape)
        
        logits = self.fc(max_vals)
        print(logits.shape)
        return logits

In [97]:
crnn = CRNN()
crnn(torch.randn(32, 1, 128, 1292))

torch.Size([32, 1, 128, 1292])
torch.Size([32, 32, 64, 646])
torch.Size([32, 64, 32, 323])
torch.Size([32, 323, 2048])
torch.Size([32, 323, 128])
torch.Size([32, 128])
torch.Size([32, 10])


tensor([[-0.5566,  0.0183, -0.1206,  0.3784, -0.6958, -0.3150,  0.3647,  0.1263,
         -0.3346, -0.2804],
        [-0.5193,  0.1781, -0.0837,  0.1932, -0.6705, -0.4150,  0.3622,  0.1338,
         -0.2554, -0.2868],
        [-0.5094,  0.1722, -0.0306,  0.2577, -0.6886, -0.3961,  0.3658,  0.0825,
         -0.2390, -0.2354],
        [-0.5419,  0.1474, -0.1737,  0.3547, -0.7416, -0.3084,  0.2298,  0.1911,
         -0.2926, -0.3784],
        [-0.4342,  0.1123, -0.1358,  0.3562, -0.5991, -0.3756,  0.3718,  0.1821,
         -0.3321, -0.2634],
        [-0.6017,  0.1042, -0.0707,  0.2959, -0.8127, -0.2841,  0.3411,  0.1572,
         -0.1740, -0.3402],
        [-0.5282,  0.1112, -0.0902,  0.2484, -0.7107, -0.3228,  0.3739,  0.1447,
         -0.2114, -0.2758],
        [-0.5571,  0.0439, -0.0288,  0.2939, -0.6734, -0.3027,  0.2739,  0.1481,
         -0.2300, -0.2342],
        [-0.4796,  0.0901, -0.0855,  0.3478, -0.6934, -0.3440,  0.3758,  0.1027,
         -0.2545, -0.1911],
        [-0.5444,  

In [99]:
sum(p.numel() for p in crnn.lstm.parameters() if p.requires_grad)

1082368

In [113]:
4*(64*(2048+64+1))*2

1081856

In [103]:
1082368-1081856

512

(Apparently, the discrepancy between the usual formula and the actual parameter count is because Pytorch's implementation of LSTMs uses 2 bias vectors per gate equation instead of the usual 1.)

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = CRNN(num_classes=10).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# test_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)